In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%pwd

'/projectnb/batmanlab/mragoza/lung-project/notebooks/bmc-4dct'

In [2]:
import sys, os

def add_path(p: str):
    if p not in sys.path:
        sys.path.append(p)

add_path(os.environ['LP_ROOT'])
import project

add_path(os.environ['PROJECT'] + '/' + 'param_search')
import param_search as ps

ps.set_backend('sge')
ps.set_verbose(False)

# Gather examples

In [3]:
from pathlib import Path
data_root = Path(os.environ['PRIVATE']) / 'data' / 'BMC-4DCT'
data_root.exists()

True

In [4]:
manifest_path = data_root / 'metadata' / 'phase_manifest.csv'
manifest = project.fileio.load_csv(manifest_path)
manifest

Loading /restricted/projectnb/batmanlab/mragoza/data/BMC-4DCT/metadata/phase_manifest.csv


,case_id,subject_num,study_num,scan_role,resp_phase,image_path,series_description,case_phases,case_valid
0,4DCT_10_1,10,1,4DCT,0.0,converted/2026-03-25_batch/4DCT/4D0010_1/4D001...,0 0% Linear,"(0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0...",True
1,4DCT_10_1,10,1,4DCT,10.0,converted/2026-03-25_batch/4DCT/4D0010_1/4D001...,10 10% Linear,"(0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0...",True
2,4DCT_10_1,10,1,4DCT,20.0,converted/2026-03-25_batch/4DCT/4D0010_1/4D001...,20 20% Linear,"(0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0...",True
3,4DCT_10_1,10,1,4DCT,30.0,converted/2026-03-25_batch/4DCT/4D0010_1/4D001...,30 30% Linear,"(0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0...",True
4,4DCT_10_1,10,1,4DCT,40.0,converted/2026-03-25_batch/4DCT/4D0010_1/4D001...,40 40% Linear,"(0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0...",True
...,...,...,...,...,...,...,...,...,...
484,unknown_53_1,53,1,unknown,50.0,converted/2026-07-16_batch/53/53_01_SBRT_4D_CT...,"Extended FOV, 50% Linear","(0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0...",True
485,unknown_53_1,53,1,unknown,60.0,converted/2026-07-16_batch/53/53_01_SBRT_4D_CT...,"Extended FOV, 60% Linear","(0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0...",True
486,unknown_53_1,53,1,unknown,70.0,converted/2026-07-16_batch/53/53_01_SBRT_4D_CT...,"Extended FOV, 70% Linear","(0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0...",True
487,unknown_53_1,53,1,unknown,80.0,converted/2026-07-16_batch/53/53_01_SBRT_4D_CT...,"Extended FOV, 80% Linear","(0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0...",True


In [5]:
case_data = manifest[manifest.case_valid].sort_values(by=['subject_num', 'study_num'])
case_list = case_data.case_id.unique().tolist()
len(case_list)

47

# Setup experiment

In [6]:
base_dir = '2026-11-01_preprocess'

template = '''\
#!/bin/bash -l
#$ -N {job_name}
#$ -P batmanlab
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_memory=32G
#$ -l h_rt=3:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/preprocess.py {config} \\
    --set dataset.subjects=[{subject}] \\
    --set dataset.variant={variant} \\
    --set dataset.pipeline_tags.image_registration={reg_tag}

'''
name_format = 'job_{params_hash}'

grid = ps.param_grid(
    config='2026-11-01_config.yaml',
    variant='2026-11-01',
    subject=case_list,
    reg_tag='gil'
)

len(grid)

47

In [7]:
base_dir = '2026-11-02_preprocess'

template = '''\
#!/bin/bash -l
#$ -N {job_name}
#$ -P batmanlab
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_memory=32G
#$ -l h_rt=3:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/preprocess.py {config} \\
    --set dataset.subjects=[{subject}] \\
    --set dataset.variant={variant} \\
    --set dataset.pipeline_tags.image_registration={reg_tag} \\
    --set preprocessing.image_registration.kwargs.model=unigradicon \\
    --set preprocessing.image_registration.kwargs.similarity=lncc \\
    --set preprocessing.image_registration.kwargs.iterations=200 \\
    --set preprocessing.image_registration.kwargs.sigma=5

'''
name_format = 'job_{params_hash}'

grid = ps.param_grid(
    config='2026-11-01_config.yaml',
    variant='2026-11-01',
    subject=case_list,
    reg_tag='ugi'
)

len(grid)

47

In [ ]:
base_dir = '2026-11-06_preprocess'

template = '''\
#!/bin/bash -l
#$ -N {job_name}
#$ -P batmanlab
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_memory=32G
#$ -l h_rt=3:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/preprocess.py {config} \\
    --set dataset.subjects=[{subject}] \\
    --set dataset.variant={variant} \\
    --set dataset.pipeline_tags.image_registration={reg_tag} \\
    --set dataset.pipeline_tags.material_properties={mat_tag}
'''
name_format = 'job_{params_hash}'

grid = ps.param_grid(
    config='2026-11-01_config.yaml',
    variant='2026-11-01',
    subject=case_list,
    reg_tag='gil',
    mat_tag=['const', 'image']
)

len(grid)

In [8]:
jobs = ps.setup(base_dir, template, name_format, grid, overwrite=False)
jobs

OSError: /projectnb/batmanlab/mragoza/lung-project/notebooks/bmc-4dct/2026-11-02_preprocess/job_0eca7b86dffc5c2c/run.sh already exists

In [ ]:
#jobs = ps.submit(jobs)

# Monitor submitted jobs

In [39]:
jobs = ps.load(base_dir)

In [40]:
jobs = ps.recover(jobs)
jobs = ps.status(jobs)
jobs = ps.collect(jobs)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.config,params.variant,params.subject,params.reg_tag,array_idx,last_live_at,state_source,output_exists,output_fsize,output_mtime
0,job_0eca7b86dffc5c2c,PENDING,1,8451394,None,None,"ICONLoss(all_loss=tensor(1.9491, device='cuda:...","self.to_file_map(**kwargs)\n File ""/proje...",/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,2026-11-01,4DCT_1_1,ugi,NaN,2026-11-05T09:58:02,status,False,<NA>,<NA>
1,job_9765c94200e1de00,PENDING,1,8451395,None,None,"ICONLoss(all_loss=tensor(1.9075, device='cuda:...","self.to_file_map(**kwargs)\n File ""/proje...",/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,2026-11-01,4DCT_2_1,ugi,NaN,2026-11-05T09:58:02,status,False,<NA>,<NA>
2,job_2d77d39a4ebf35f3,PENDING,1,8451396,None,None,"ICONLoss(all_loss=tensor(1.9135, device='cuda:...",WARNING: Interpolation points outside domain: ...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,2026-11-01,4DCT_2_2,ugi,NaN,2026-11-05T09:58:02,status,False,<NA>,<NA>
3,job_50a482a580ffbe27,PENDING,1,8451397,None,None,"ICONLoss(all_loss=tensor(1.9605, device='cuda:...","self.to_file_map(**kwargs)\n File ""/proje...",/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,2026-11-01,4DCT_4_1,ugi,NaN,2026-11-05T09:58:02,status,False,<NA>,<NA>
4,job_328c7defc34c3030,PENDING,1,8451398,None,None,"ICONLoss(all_loss=tensor(1.9513, device='cuda:...","File ""/projectnb/batmanlab/mragoza/lung-proj...",/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,2026-11-01,4DCT_6_1,ugi,NaN,2026-11-05T09:58:02,status,False,<NA>,<NA>
5,job_746931f7889cbea5,PENDING,1,8451399,None,None,"ICONLoss(all_loss=tensor(1.9295, device='cuda:...",WARNING: Interpolation points outside domain: ...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,2026-11-01,4DCT_7_3,ugi,NaN,2026-11-05T09:58:02,status,False,<NA>,<NA>
6,job_a0b48b23302ee2fe,PENDING,1,8451400,None,None,"ICONLoss(all_loss=tensor(1.9159, device='cuda:...","File ""/projectnb/batmanlab/mragoza/lung-proj...",/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,2026-11-01,4DCT_8_1,ugi,NaN,2026-11-05T09:58:02,status,False,<NA>,<NA>
7,job_1db2bf509c795bce,PENDING,1,8451401,None,None,"ICONLoss(all_loss=tensor(1.9632, device='cuda:...",WARNING: Interpolation points outside domain: ...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,2026-11-01,4DCT_8_2,ugi,NaN,2026-11-05T09:58:02,status,False,<NA>,<NA>
8,job_f792bdc4de5de080,PENDING,1,8451402,None,None,"ICONLoss(all_loss=tensor(1.9345, device='cuda:...",WARNING: Interpolation points outside domain: ...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,2026-11-01,4DCT_10_1,ugi,NaN,2026-11-05T09:58:02,status,False,<NA>,<NA>
9,job_b8991533ad1958c2,PENDING,1,8451403,None,None,"ICONLoss(all_loss=tensor(1.9494, device='cuda:...",WARNING: Interpolation points outside domain: ...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,2026-11-01,unknown_12_1,ugi,NaN,2026-11-05T09:58:02,status,False,<NA>,

In [41]:
jobs.groupby('job_state').size()

job_state
PENDING    47
dtype: int64

In [50]:
import pandas as pd

def classify_output(stdout):
    if pd.isna(stdout):
        return 'No stdout file'
    elif not str(stdout).strip():
        return 'No output message'
    elif 'Done' in stdout:
        return 'Complete'
    return 'Incomplete'

def classify_error(stderr):
    if pd.isna(stderr):
        return 'No stderr file'
    elif not str(stderr).strip():
        return 'No error message'
    elif 'No space left on device' in stderr:
        return 'No storage space'
    elif 'WARNING: Interpolation points outside domain' in stderr:
        return 'Interpolation warning'
    return 'Uncategorized'

jobs['output_type'] = jobs['stdout'].map(classify_output)
jobs['error_type']  = jobs['stderr'].map(classify_error)

jobs.groupby(['output_type', 'error_type'])[['job_name']].count()

,,job_name
output_type,error_type,
Complete,Interpolation warning,42
Incomplete,No storage space,5


In [51]:
query_jobs = jobs.query('error_type == "No storage space"')
job = query_jobs.iloc[0]
display(job)
print(job.stdout)
print(job.stderr)

job_name                                       job_0eca7b86dffc5c2c
job_state                                                   PENDING
n_submits                                                         1
job_id                                                      8451394
node_id                                                        None
runtime                                                        None
stdout            ICONLoss(all_loss=tensor(1.9491, device='cuda:...
stderr                self.to_file_map(**kwargs)\n  File "/proje...
base_dir          /projectnb/batmanlab/mragoza/lung-project/note...
work_dir          /projectnb/batmanlab/mragoza/lung-project/note...
script_path       /projectnb/batmanlab/mragoza/lung-project/note...
output_path       /projectnb/batmanlab/mragoza/lung-project/note...
log_dir           /projectnb/batmanlab/mragoza/lung-project/note...
stdout_path       /projectnb/batmanlab/mragoza/lung-project/note...
stderr_path       /projectnb/batmanlab/mragoza/l

ICONLoss(all_loss=tensor(1.9491, device='cuda:0', grad_fn=<AddBackward0>), inverse_consistency_loss=tensor(0.0009, device='cuda:0', grad_fn=<AddBackward0>), similarity_loss=tensor(1.9476, device='cuda:0', grad_fn=<AddBackward0>), transform_magnitude=tensor(4.0024e-06, device='cuda:0', grad_fn=<MeanBackward0>), flips=tensor(0., device='cuda:0'))
ICONLoss(all_loss=tensor(1.9490, device='cuda:0', grad_fn=<AddBackward0>), inverse_consistency_loss=tensor(0.0009, device='cuda:0', grad_fn=<AddBackward0>), similarity_loss=tensor(1.9476, device='cuda:0', grad_fn=<AddBackward0>), transform_magnitude=tensor(3.9885e-06, device='cuda:0', grad_fn=<MeanBackward0>), flips=tensor(0., device='cuda:0'))
ICONLoss(all_loss=tensor(1.9490, device='cuda:0', grad_fn=<AddBackward0>), inverse_consistency_loss=tensor(0.0009, device='cuda:0', grad_fn=<AddBackward0>), similarity_loss=tensor(1.9476, device='cuda:0', grad_fn=<AddBackward0>), transform_magnitude=tensor(3.9911e-06, device='cuda:0', grad_fn=<MeanBackwar

In [55]:
jobs.loc[jobs.output_type == 'Incomplete', 'job_id'] = pd.NA
jobs = ps.submit(jobs)

# Validate deformation fields

In [21]:
%%time
import numpy as np

def apply_deformation(array, disp, affine, binary=False):
    from scipy.ndimage import map_coordinates

    array = np.asarray(array)
    disp = np.asarray(disp)

    if disp.shape != (*array.shape, 3):
        raise ValueError(f'Shape mismatch: {disp.shape}  vs. {array.shape}')

    world_to_voxel = np.linalg.inv(affine[:3,:3])
    disp_vox = np.einsum('ij,...j->...i', world_to_voxel, disp)

    grid = np.indices(array.shape, dtype=np.float32)
    coords = grid + np.moveaxis(disp_vox, -1, 0)

    if binary:
        return map_coordinates(
            array.astype(np.uint8),
            coords,
            order=0,
            mode='constant',
            cval=0
        )
    else:
        return map_coordinates(
            array,
            coords,
            order=1,
            mode='constant',
            cval=-1000
        )

def check_registration(ex):
    from project.evaluation import metrics as eval_metrics
    import pandas as pd

    disp_path = ex.paths['disp_field']
    nifti = project.fileio.load_nibabel(disp_path)
    disp, affine = nifti.get_fdata(), nifti.affine

    init_image_path = ex.paths['init_state']['resampled_image']
    curr_image_path = ex.paths['curr_state']['resampled_image']   

    init_mask_path = ex.paths['init_state']['combined_mask']
    curr_mask_path = ex.paths['curr_state']['combined_mask']

    init_image = project.fileio.load_nibabel(init_image_path).get_fdata()
    curr_image = project.fileio.load_nibabel(curr_image_path).get_fdata()

    init_mask = project.fileio.load_nibabel(init_mask_path).get_fdata() > 0
    curr_mask = project.fileio.load_nibabel(curr_mask_path).get_fdata() > 0

    warp_image = apply_deformation(curr_image, disp, affine, False)
    warp_mask  = apply_deformation(curr_mask, disp, affine, True)

    dice_before = eval_metrics.dice_score(curr_mask, init_mask)
    dice_after  = eval_metrics.dice_score(warp_mask, init_mask)

    corr_before = eval_metrics.pearson_corr(curr_image, init_image, init_mask)
    corr_after  = eval_metrics.pearson_corr(warp_image, init_image, init_mask)

    return {
        'dice_before': dice_before,
        'dice_after':  dice_after,
        'corr_before': corr_before,
        'corr_after':  corr_after
    }

def _load_examples(reg_tag):
    config = project.fileio.load_config('2026-11-01_config.yaml')
    config['dataset']['subjects'] = case_list
    config['dataset']['pipeline_tags']['image_registration'] = reg_tag
    return project.get_examples(config['dataset'])

examples_by_reg = { # first 100 takes ~3 hours
    'gil': _load_examples(reg_tag='gil')[:100],
    'ugi': _load_examples(reg_tag='ugi')[:100]
}

project.utils.set_verbose(False)

rows = []
for reg_tag, examples in examples_by_reg.items():
    for ex in examples:
        try:
            vals = check_registration(ex)
        except FileNotFoundError as e:
            print(e)
            continue
    
        row = {
            'case_id': ex.subject,
            'init_phase': ex.metadata['init_state'],
            'curr_phase': ex.metadata['curr_state'],
            'reg_tag': reg_tag,
            **vals
        }
        print(row)
        rows.append(row)

df = pd.DataFrame(rows)
df.to_csv('registration_metrics.csv')
df

Loading 2026-11-01_config.yaml
Gathering examples...
423 total
Loading 2026-11-01_config.yaml
Gathering examples...
423 total
{'case_id': '4DCT_1_1', 'init_phase': 50, 'curr_phase': 0, 'reg_tag': 'gil', 'dice_before': 0.8441080755138489, 'dice_after': 0.9699312812982251, 'corr_before': 0.33447277978142054, 'corr_after': 0.9190660425046695}
{'case_id': '4DCT_1_1', 'init_phase': 50, 'curr_phase': 10, 'reg_tag': 'gil', 'dice_before': 0.8722461242003732, 'dice_after': 0.9710089833698089, 'corr_before': 0.4051052523083011, 'corr_after': 0.9223518085131522}
{'case_id': '4DCT_1_1', 'init_phase': 50, 'curr_phase': 20, 'reg_tag': 'gil', 'dice_before': 0.9188577774068893, 'dice_after': 0.9709779758559562, 'corr_before': 0.5583268839541461, 'corr_after': 0.9406109571411567}
{'case_id': '4DCT_1_1', 'init_phase': 50, 'curr_phase': 30, 'reg_tag': 'gil', 'dice_before': 0.9595284823379432, 'dice_after': 0.9792886152869219, 'corr_before': 0.755939319407162, 'corr_after': 0.9662125152155084}
{'case_id':

,case_id,init_phase,curr_phase,reg_tag,dice_before,dice_after,corr_before,corr_after
0,4DCT_1_1,50,0,gil,0.844108,0.969931,0.334473,0.919066
1,4DCT_1_1,50,10,gil,0.872246,0.971009,0.405105,0.922352
2,4DCT_1_1,50,20,gil,0.918858,0.970978,0.558327,0.940611
3,4DCT_1_1,50,30,gil,0.959528,0.979289,0.755939,0.966213
4,4DCT_1_1,50,40,gil,0.983644,0.987058,0.927895,0.981422
...,...,...,...,...,...,...,...,...
172,unknown_13_1,50,60,ugi,0.984185,0.989341,0.852779,0.956188
173,unknown_13_1,50,70,ugi,0.969667,0.984055,0.699125,0.936045
174,unknown_13_1,50,80,ugi,0.953206,0.980646,0.580784,0.918818
175,unknown_13_1,50,90,ugi,0.944883,0.980250,0.522190,0.914920


In [22]:
df

,case_id,init_phase,curr_phase,reg_tag,dice_before,dice_after,corr_before,corr_after
0,4DCT_1_1,50,0,gil,0.844108,0.969931,0.334473,0.919066
1,4DCT_1_1,50,10,gil,0.872246,0.971009,0.405105,0.922352
2,4DCT_1_1,50,20,gil,0.918858,0.970978,0.558327,0.940611
3,4DCT_1_1,50,30,gil,0.959528,0.979289,0.755939,0.966213
4,4DCT_1_1,50,40,gil,0.983644,0.987058,0.927895,0.981422
...,...,...,...,...,...,...,...,...
172,unknown_13_1,50,60,ugi,0.984185,0.989341,0.852779,0.956188
173,unknown_13_1,50,70,ugi,0.969667,0.984055,0.699125,0.936045
174,unknown_13_1,50,80,ugi,0.953206,0.980646,0.580784,0.918818
175,unknown_13_1,50,90,ugi,0.944883,0.980250,0.522190,0.914920


In [38]:
df['dice_delta'] = df['dice_after'] - df['dice_before']
df['corr_delta'] = df['corr_after'] - df['corr_before']

df['second_batch'] = df['case_id'].str.startswith('unknown')

df.groupby(['second_batch', 'reg_tag'])[['dice_delta', 'corr_delta']].agg(['count', 'mean', 'median', 'min', 'max'])

dice_delta                                          \
                          count      mean    median       min       max   
second_batch reg_tag                                                      
False        gil             81  0.023495  0.014512 -0.000235  0.125823   
             ugi             58  0.021435  0.014643 -0.004569  0.128339   
True         gil             19  0.018211  0.013819  0.001668  0.039775   
             ugi             19  0.019296  0.014646  0.002107  0.040584   

                     corr_delta                                          
                          count      mean    median       min       max  
second_batch reg_tag                                                     
False        gil             81  0.178279  0.139718  0.012396  0.584593  
             ugi             58  0.150928  0.107563 -0.004263  0.571052  
True         gil             19  0.248342  0.255864  0.040988  0.441751  
             ugi             19  0.232708  0.238666  0.039463  0.413036

In [35]:
paired = df.pivot(
    index=['case_id', 'init_phase', 'curr_phase'],
    columns='reg_tag',
    values=['dice_delta', 'corr_delta']
)
delta = paired.xs('gil', axis=1, level='reg_tag') - paired.xs('ugi', axis=1, level='reg_tag')

summary = delta.agg(['count', 'mean', 'median', 'min', 'max']).T
summary['gil_better'] = delta.gt(0).mean()
summary['ugi_better'] = delta.lt(0).mean()
summary['tied'] = delta.eq(0).mean()
summary['null'] = delta.isna().mean()

summary

,count,mean,median,min,max,gil_better,ugi_better,tied,null
dice_delta,77.0,0.000359,-0.000389,-0.002516,0.011164,0.26,0.51,0.0,0.23
corr_delta,77.0,0.016501,0.014479,-0.000923,0.056273,0.75,0.02,0.0,0.23
